In [1]:
import os
import random
import time
import h5py
import numpy as np
import pandas as pd
import scipy.io
import librosa
import kagglehub
import h5py
import shutil
import IPython.display as ipd

from IPython.display import display, Audio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score 
from sktime.classification.kernel_based import RocketClassifier
from sktime.transformations.panel.rocket import Rocket
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, classification_report
from detach_rocket.detach_classes import DetachRocket


In [2]:
# ==================================================
# 2. CARGA Y CONCATENACIÓN DE BABBLE NOISE (LOCAL)
# ==================================================
babble_exact_path = "datasets/noise/0dB/valid/real" 

audio_buffers_list = []
max_archivos_a_combinar = 40 
sr_original = 16000
sr_objetivo = 22050

archivos_mat = 0

for f in os.listdir(babble_exact_path):
    if len(audio_buffers_list) >= max_archivos_a_combinar:
        break

    file_path = os.path.join(babble_exact_path, f)
    
    if file_path.endswith('.mat'):
            mat_contents = scipy.io.loadmat(file_path) #esta funcion convierte el .mat en un diccionario 
            keys = [k for k in mat_contents.keys() if not k.startswith('__')] # cuando SciPy lee un .mat, automáticamente le inyecta variables internas de configuración que siempre empiezan con doble guion bajo (ej: __header. 
            #armo una lista solo con los nombres de las variables "reales" que contienen el audio
            if keys:
                audio_raw = mat_contents[keys[0]].flatten().astype(np.float32) 
                #mat_contents[keys[0]] extrae los datos almacenados en la primera clave válida
                # .flatten(): "aplasta" esa matriz para convertirla en un vector 1D
                # .astype(np.float32): conversión a formato decimal de 32 bits (float32).
                if audio_raw.size > 0:
                    audio_buffers_list.append(audio_raw)
                    archivos_mat += 1

# Procesamiento final
print("-" * 50)
print(f" - Archivos .mat procesados: {archivos_mat}")



--------------------------------------------------
 - Archivos .mat procesados: 40


In [3]:
babble_completo_16k = np.concatenate(audio_buffers_list)
babble_audio_full = librosa.resample(babble_completo_16k, orig_sr=sr_original, target_sr=sr_objetivo) #la paso a la sample rate que necesito

In [4]:
# ==================================================
# 3. FUNCIONES DE DATA AUGMENTATION
# ==================================================

#RUIDO BLANCO
def add_white_noise(audio, noise_level=0.005):
    noise = np.random.randn(len(audio)) #vector de numeros aleatorios con la misma longitud que mi vector original
    return audio + noise_level * noise

In [5]:
#RUIDO ROSA
def add_pink_noise(audio, noise_level=0.01):
    #el ruido rosa se caracteriza por tener una densidad espectral inversa a la frecuencia (es decir, 1/f)
    white = np.random.randn(len(audio)) #genero ruido blanco
    fft_white = np.fft.rfft(white) #transformada de fourier - paso el ruido blanco a dominio de la frecuencia. Ya no es la potencia, es la amplitud la que veo representada
    frequencies = np.maximum(np.fft.rfftfreq(len(audio)), 1e-10) #crea un vector de frecuencias. SI hay un valor menor a 1e-10, lo reemplaza por este número.
    #Elijo 1e-10 porque es un numero cercano a cero, sin ser cero.
    f_filter = 1.0 / np.sqrt(frequencies) #En la línea anterior, me aseguré de que no queden divisiones por cero. V^2 = P -> 1/sqrt(f) = P (ver notas)
    f_filter /= np.max(f_filter) #Busca el numero mas grande que haya quedado dentro del array y divide a todos los valores por este numero
    fft_pink = fft_white * f_filter #Aplica el filtro al ruido blanco en el dominio de la frecuencia.
    pink = np.fft.irfft(fft_pink, n=len(audio)) #Conversion al dominio del tiempo
    pink = pink / np.max(np.abs(pink)) #normalizacion
    return audio + noise_level * pink

In [6]:

#MULTITALKER BABBLE NOISE 
def add_babble_noise(audio, babble_audio, noise_level=0.03):
    # np.tile -> construye un nuevo array repitiendo el primer argumento (en este caso, babble_audio) la cantidad de veces que se pida. 
    # Esto se aplica solo si el audio es menor al audio original  
    #np.ceil -> devuelve el menor escalar i tal que i>=x
    if len(babble_audio) < len(audio):
        babble_audio = np.tile(babble_audio, int(np.ceil(len(audio) / len(babble_audio))))

    # toma un punto de inicio random
    start_idx = random.randint(0, len(babble_audio) - len(audio))
    babble_chunk = babble_audio[start_idx : start_idx + len(audio)] #aplico babble noise desde mi punto random hasta el final del audio
    babble_chunk = babble_chunk / (np.max(np.abs(babble_chunk)) + 1e-10) #normalizo el murmullo para que no haya picos de volumen descontrolados

    return audio + noise_level * babble_chunk 


In [7]:
# ==========================================
# 4. DESCARGA Y CURACIÓN DEL DATASET LOCAL
# ==========================================
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

metadata_path = None
for root, dirs, files in os.walk(dataset_root_path):
    if 'esc50.csv' in files:
        metadata_path = os.path.join(root, 'esc50.csv')
        break

if metadata_path:
    df = pd.read_csv(metadata_path)
    print("Archivo de metadatos cargado correctamente.")

mis_clases = ['clock_alarm', 'door_wood_knock', 'cat', 'crying_baby', 'dog', 'glass_breaking']
df_filtrado = df[df['category'].isin(mis_clases)].copy()

kaggle_audio_dir = None
for root, dirs, files in os.walk(dataset_root_path): #Busca donde estan los archivos
    if any(f.endswith('.wav') for f in files):
        kaggle_audio_dir = root
        break

target_dir = "datasets/audios_originales"
archivos_copiados = 0

for index, row in df_filtrado.iterrows():
    src_path = os.path.join(kaggle_audio_dir, row['filename'])
    dst_path = os.path.join(target_dir, row['filename'])
    
    # Solo lo copio si el archivo original existe y no esta en el directorio audios_originales
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        shutil.copy2(src_path, dst_path)
        archivos_copiados += 1

print(f"Se copiaron {archivos_copiados} archivos nuevos.")


Archivo de metadatos cargado correctamente.
Se copiaron 0 archivos nuevos.


In [8]:
# ==========================================
# 4. DESCARGA Y CURACIÓN DEL DATASET LOCAL
# ==========================================
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

metadata_path = None
for root, dirs, files in os.walk(dataset_root_path):
    if 'esc50.csv' in files:
        metadata_path = os.path.join(root, 'esc50.csv')
        break

if metadata_path:
    df = pd.read_csv(metadata_path)

In [9]:
# COPIO SOLO LAS CLASES QUE ME INTERESAN

mis_clases = ['clock_alarm', 'door_wood_knock', 'cat', 'crying_baby', 'dog', 'glass_breaking']
df_filtrado = df[df['category'].isin(mis_clases)].copy()

In [10]:
#COPIO SOLOS LOS ARCHIVOS NUEVOS

target_dir = "datasets/audios_originales"
archivos_copiados = 0

for index, row in df_filtrado.iterrows():
    src_path = os.path.join(kaggle_audio_dir, row['filename'])
    dst_path = os.path.join(target_dir, row['filename'])
    
    # Solo lo copio si el archivo original existe y no esta en el directorio audios_originales
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        shutil.copy2(src_path, dst_path)
        archivos_copiados += 1

print(f"Se copiaron {archivos_copiados} archivos nuevos.")

Se copiaron 0 archivos nuevos.


In [11]:
# ==========================================
# 5. TRANSFORMACIÓN Y APLICO DATA AUGMENTATION
# ==========================================
correct_audio_dir = target_dir 

X_list = []
y_list = []
groups = []


for index, row in df_filtrado.iterrows(): #iterrows devuelve el indice y el contenido de la fila 
    file_path = os.path.join(correct_audio_dir, row['filename']) #ejemplo: datasets/audios_originales/1-211527-C-20.wav
    categoria = row['category'] #ejemplo: crying_baby
    
    y_audio, sr = librosa.load(file_path, sr=22050) #lee audio y lo convierte

    audios_a_procesar = [
            y_audio,                                     
            add_white_noise(y_audio),                    
            add_pink_noise(y_audio),                     
            add_babble_noise(y_audio, babble_audio_full) 
        ]

    for audio_version in audios_a_procesar:
        X_list.append(audio_version)
        y_list.append(categoria)
        groups.append(row['filename'])

X = np.array(X_list)
y = np.array(y_list)
groups = np.array(groups)

print("-" * 50)
print(f"Forma final de la matriz X: {X.shape}")
print(f"Cantidad de etiquetas y: {y.shape}")


--------------------------------------------------
Forma final de la matriz X: (960, 110250)
Cantidad de etiquetas y: (960,)


In [12]:
groups
y

array(['dog', 'dog', 'dog', 'dog', 'door_wood_knock', 'door_wood_knock',
       'door_wood_knock', 'door_wood_knock', 'door_wood_knock',
       'door_wood_knock', 'door_wood_knock', 'door_wood_knock',
       'door_wood_knock', 'door_wood_knock', 'door_wood_knock',
       'door_wood_knock', 'dog', 'dog', 'dog', 'dog', 'clock_alarm',
       'clock_alarm', 'clock_alarm', 'clock_alarm', 'clock_alarm',
       'clock_alarm', 'clock_alarm', 'clock_alarm', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'glass_breaking',
       'glass_breaking', 'glass_breaking', 'glass_breaking',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'crying_baby', 'crying_baby', 'crying_baby', 'crying_baby',
       'door_wood_knock', 'door_wood_knock', 'door_wood_knock',
  

In [13]:
# ==========================================
# 6.1 ROCKET
# ==========================================


le = LabelEncoder() #convierte text labels en numeros del 0 a nro_de_clases - 1
y_encoded = le.fit_transform(y)

gkf = GroupKFold(n_splits=5) # me aseguro de que el audio original y sus deformaciones queden todos en train/test

accuracies = [] #guardo resultado de cada fold para despues hacer un promedio

print("-" * 50)
print(f"Total de muestras aumentadas: {len(X)}")

for fold, (train_index, test_index) in enumerate(gkf.split(X, y_encoded, groups=groups)): #gkf.split mira los datos, las etiquetas y los identificadores del archivo original y decide de forma inteligente que numeros de fila se usaran para train y test
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y_encoded[train_index], y_encoded[test_index]
    
    rocket_model = RocketClassifier(num_kernels=500)
    
    # Entrenamiento
    rocket_model.fit(X_train, y_train)
    
    # Evaluación
    y_pred = rocket_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    
    print(f" - Fold {fold+1}: Accuracy = {acc:.4f}")

# Resultado final (promedio de folds)
print("-" * 50)
print(f"Accuracy Promedio (Validado): {np.mean(accuracies):.4f}")

#TIEMPO DE EJECUCION: 32 minutos

--------------------------------------------------
Total de muestras aumentadas: 960


KeyboardInterrupt: 

In [14]:
X_train


array([[ 0.        , -0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.0003363 , -0.0054085 , -0.00486046, ..., -0.00745608,
         0.00535158, -0.00319868],
       [ 0.00945212,  0.00956458,  0.00965402, ...,  0.00941727,
         0.00963107,  0.00956423],
       ...,
       [-0.00251884,  0.00753264,  0.00290672, ..., -0.00056518,
         0.00156895, -0.0023112 ],
       [ 0.0135203 ,  0.01821903,  0.01335669, ...,  0.00933549,
         0.00936651,  0.00943634],
       [ 0.00359004,  0.00805729,  0.0035005 , ..., -0.00056578,
        -0.00013202, -0.0001103 ]])

In [ ]:
import time
import numpy as np
from sklearn.metrics import accuracy_score

accuracies = []

print(f"Total de muestras: {len(X)}")

for fold, (train_index, test_index) in enumerate(gkf.split(X, y_encoded, groups=groups)): 
    print(f"\n INICIANDO FOLD {fold+1} DE 5")
    
    # solo consegui que funcionara agregandole esta dimension 
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y_encoded[train_index], y_encoded[test_index]
    
    X_train_3d = np.expand_dims(X_train, axis=1) 
    X_test_3d = np.expand_dims(X_test, axis=1)
    
    # Instancio modelo
    kernels_a_usar = 1000 #10000 se traba
    DetachRocketModel = DetachRocket('rocket', num_kernels=kernels_a_usar)
    
    # Guardo log
    hora_inicio_entrenamiento = time.strftime('%H:%M:%S')
    print(f"   [{hora_inicio_entrenamiento}] Entrenando DetachRocket con {kernels_a_usar} kernels...")
    
    start_time = time.time()
    DetachRocketModel.fit(X_train_3d, y_train)
    tiempo_entrenamiento = (time.time() - start_time) / 60
    
    print(f"Entrenamiento listo. Tardó: {tiempo_entrenamiento:.2f} minutos.")
    
    # Predicción y Evaluación
    y_pred = DetachRocketModel.predict(X_test_3d)
    
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    
    print(f"Resultado Fold {fold+1}: Accuracy = {acc:.4f}")

# Resultado final
print("\n" + "=" * 50)
print(f"ACCURACY PROMEDIO FINAL: {np.mean(accuracies):.4f}")
print("=" * 50)

#Tiempo de ejecucion 25 minutos aprox

Total de muestras: 960

▶️ INICIANDO FOLD 1 DE 5...
   [22:44:22] Entrenando DetachRocket con 1000 kernels...


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


TRAINING RESULTS Full ROCKET:
Optimal Alpha Full ROCKET: 0.03
Train Accuraccy Full ROCKET: 100.00%
-------------------------


/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=2.73247e-09): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=3.29715e-09): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=2.56009e-09): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:253: LinAlgWarning: Ill-conditioned matrix (rcond=3.60245e-09): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=Fal

TRAINING RESULTS Detach Model:
Optimal Alpha Detach Model: 0.30
Train Accuraccy Detach Model: 99.87%
-------------------------
   ✅ Entrenamiento listo. Tardó: 4.64 minutos.
   ⏳ Evaluando Test Set...
🎯 Resultado Fold 1: Accuracy = 0.8333

▶️ INICIANDO FOLD 2 DE 5...
   [22:50:11] Entrenando DetachRocket con 1000 kernels...
TRAINING RESULTS Full ROCKET:
Optimal Alpha Full ROCKET: 0.00
Train Accuraccy Full ROCKET: 100.00%
-------------------------


...

TRAINING RESULTS Detach Model:
Optimal Alpha Detach Model: 0.03
Train Accuraccy Detach Model: 100.00%
-------------------------
   ✅ Entrenamiento listo. Tardó: 4.70 minutos.
   ⏳ Evaluando Test Set...
🎯 Resultado Fold 3: Accuracy = 0.6927

▶️ INICIANDO FOLD 4 DE 5...
   [23:01:58] Entrenando DetachRocket con 1000 kernels...
TRAINING RESULTS Full ROCKET:
Optimal Alpha Full ROCKET: 0.00
Train Accuraccy Full ROCKET: 100.00%
-------------------------


/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution i

TRAINING RESULTS Detach Model:
Optimal Alpha Detach Model: 0.03
Train Accuraccy Detach Model: 100.00%
-------------------------
   ✅ Entrenamiento listo. Tardó: 4.73 minutos.
   ⏳ Evaluando Test Set...
🎯 Resultado Fold 4: Accuracy = 0.8229

▶️ INICIANDO FOLD 5 DE 5...
   [23:07:53] Entrenando DetachRocket con 1000 kernels...
TRAINING RESULTS Full ROCKET:
Optimal Alpha Full ROCKET: 0.00
Train Accuraccy Full ROCKET: 100.00%
-------------------------


/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution i

TRAINING RESULTS Detach Model:
Optimal Alpha Detach Model: 0.30
Train Accuraccy Detach Model: 100.00%
-------------------------
   ✅ Entrenamiento listo. Tardó: 4.75 minutos.
   ⏳ Evaluando Test Set...
🎯 Resultado Fold 5: Accuracy = 0.5938

🚀 ACCURACY PROMEDIO FINAL: 0.7292


In [ ]:
# ==========================================
# 6.2 DETACH ROCKET 
# ==========================================

le = LabelEncoder() # convierte text labels en numeros del 0 a nro_de_clases - 1
y_encoded = le.fit_transform(y)

gkf = GroupKFold(n_splits=5) # me aseguro de que el audio original y sus deformaciones queden todos en train/test

accuracies = [] # guardo resultado de cada fold para despues hacer un promedio

print("-" * 50)
print(f"Total de muestras aumentadas: {len(X)}")

# El bucle de validación cruzada
for fold, (train_index, test_index) in enumerate(gkf.split(X, y_encoded, groups=groups)): 
    # 1. Expandimos las dimensiones de 2D a 3D (n_instances, n_channels, n_timepoints)
    X_train_3d = np.expand_dims(X_train, axis=1)
    X_test_3d = np.expand_dims(X_test, axis=1)

    # 2. Instantiate Model
    DetachRocketModel = DetachRocket('rocket', num_kernels=10000)

    # 3. Train Model (Pasamos la nueva matriz 3D)
    DetachRocketModel.fit(X_train_3d, y_train)

    # 4. Predict Test Set
    y_pred = DetachRocketModel.predict(X_test_3d)
    
    # Evaluación
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    
    print(f" - Fold {fold+1}: Accuracy = {acc:.4f}")

# 4. Resultado final (la media de todos los folds)
print("-" * 50)
print(f"Accuracy Promedio (Validado): {np.mean(accuracies):.4f}")

--------------------------------------------------
Total de muestras aumentadas: 960


In [ ]:
# ==========================================
# 6.2 DETACH ROCKET (La de arriba funciona mejor)
# ==========================================

le = LabelEncoder() # convierte text labels en numeros del 0 a nro_de_clases - 1
y_encoded = le.fit_transform(y)

gkf = GroupKFold(n_splits=5) # me aseguro de que el audio original y sus deformaciones queden todos en train/test

accuracies = [] # guardo resultado de cada fold para despues hacer un promedio

print("-" * 50)
print(f"Total de muestras aumentadas: {len(X)}")


def to_nested_df(X_numpy):
    X_2d = np.squeeze(X_numpy) # Nos aseguramos de aplanar a formato (N, 40)
    df = pd.DataFrame()
    # Metemos cada vector de 40 características en una celda como una Serie de Pandas
    df['dim_0'] = [pd.Series(row) for row in X_2d] 
    return df

for fold, (train_index, test_index) in enumerate(gkf.split(X, y_encoded, groups=groups)): 
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y_encoded[train_index], y_encoded[test_index]
    
    # Aplicamos la conversión a nuestros sets
    X_train_nested = to_nested_df(X_train)
    X_test_nested = to_nested_df(X_test)
    
    # Instantiate Model
    DetachRocketModel = DetachRocket('rocket', num_kernels=5000)
    
    # Train Model (¡Atención! Pasamos el Dataframe Anidado)
    DetachRocketModel.fit(X_train_nested, y_train)
    
    # Predict Test Set
    y_pred = DetachRocketModel.predict(X_test_nested)
    
    # Evaluación
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    
    print(f" - Fold {fold+1}: Accuracy = {acc:.4f}")

# 4. Resultado final (la media de todos los folds)
print("-" * 50)
print(f"Accuracy Promedio (Validado): {np.mean(accuracies):.4f}")

--------------------------------------------------
Total de muestras aumentadas: 960


In [ ]:
<<PLAYGROUND>>

In [ ]:
# ==========================================
# 5. DETACH ROCKET 
# ==========================================


le = LabelEncoder() #convierte text labels en numeros del 0 a nro_de_clases - 1
y_encoded = le.fit_transform(y)

gkf = GroupKFold(n_splits=5) # me aseguro de que el audio original y sus deformaciones queden todos en train/test

accuracies = [] #guardo resultado de cada fold para despues hacer un promedio

print("-" * 50)
print(f"Total de muestras aumentadas: {len(X)}")

for fold, (train_index, test_index) in enumerate(gkf.split(X, y_encoded, groups=groups)): #gkf.split mira los datos, las etiquetas y los identificadores del archivo original y decide de forma inteligente que numeros de fila se usaran para train y test
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y_encoded[train_index], y_encoded[test_index]
    
model_type = "rocket"
num_kernels = 10000

DetachRocketModel = DetachRocket(model_type, num_kernels=num_kernels)
# 4. Resultado final (la media de todos los folds)
print("-" * 50)
print(f"Accuracy Promedio (Validado): {np.mean(accuracies):.4f}")

--------------------------------------------------
Total de muestras aumentadas: 960
--------------------------------------------------
Accuracy Promedio (Validado): nan


/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/miniconda3/envs/pfi_audio/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
# ==========================================
# 5. ENTRENAMIENTO CON DETACH-ROCKET
# ==========================================
from sklearn.ensemble import RandomForestClassifier
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("-" * 50)
print("RESUMEN DE AUDIOS UTILIZADOS:")
print(f"Total de muestras (con Data Augmentation): {len(X)}")
print(f" - Muestras para Entrenamiento: {len(X_train)}")
print(f" - Muestras para Prueba (Test): {len(X_test)}")
print("-" * 50)

inicio = time.time()

# Instanciar y ajustar Detach-ROCKET
detach_model = RocketClassifier(num_kernels=500)
detach_model.fit(X_train, y_train)

--------------------------------------------------
RESUMEN DE AUDIOS UTILIZADOS:
Total de muestras (con Data Augmentation): 480
 - Muestras para Entrenamiento: 384
 - Muestras para Prueba (Test): 96
--------------------------------------------------


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


RocketClassifier(num_kernels=500)

In [ ]:

# Hacer predicciones sobre el conjunto de prueba
y_pred = detach_model.predict(X_test)

# Calcular la métrica con sklearn
accuracy = accuracy_score(y_test, y_pred)


In [ ]:
accuracy

1.0

In [ ]:



# Hacer predicciones sobre el conjunto de prueba
y_pred = detach_model.predict(X_test)

# Calcular la métrica con sklearn
accuracy = accuracy_score(y_test, y_pred)
fin = time.time()

print(f"Precisión (Test Accuracy): {accuracy * 100:.2f}%")
print(f"Tiempo total de ejecución: {fin - inicio:.2f} segundos")

# DetachRocket permite visualizar qué proporción de variables (features) retuvo:
if hasattr(detach_model, 'feature_proportion_'):
    print(f"Porcentaje de features retenidas tras el podado: {detach_model.feature_proportion_ * 100:.2f}%")

# ==========================================
# 6. AUDICIÓN DE VARIANTES
# ==========================================
if 'df_filtrado' in locals() and 'correct_audio_dir' in locals() and correct_audio_dir is not None:
    random_row = df_filtrado.sample(n=1).iloc[0]
    random_file_path = os.path.join(correct_audio_dir, random_row['filename'])
    random_label = random_row['category']

    print("=" * 50)
    print(f"AUDICIÓN DE VARIANTES: {random_label.upper()}")
    print(f"Archivo base: {random_row['filename']}")
    print("=" * 50)

    try:
        y_audio_test, sr_audio = librosa.load(random_file_path, sr=22050)
        
        print("\n1. Audio Original:")
        display(ipd.Audio(y_audio_test, rate=sr_audio))
        
        print("\n2. Variante: Ruido Blanco:")
        y_white = add_white_noise(y_audio_test)
        display(ipd.Audio(y_white, rate=sr_audio))
            
        print("\n3. Variante: Ruido Rosa:")
        y_pink = add_pink_noise(y_audio_test)
        display(ipd.Audio(y_pink, rate=sr_audio))
            
        if babble_audio_full is not None:
            print("\n4. Variante: Murmullo de fondo (Babble Noise):")
            y_babble = add_babble_noise(y_audio_test, babble_audio_full)
            display(ipd.Audio(y_babble, rate=sr_audio))
        else:
            print("\n[Aviso: No se generó Babble Noise válido]")
        
    except Exception as e:
        print(f"Error interno al intentar procesar los audios: {e}")

--------------------------------------------------
RESUMEN DE AUDIOS UTILIZADOS:
Total de muestras (con Data Augmentation): 480
 - Muestras para Entrenamiento: 384
 - Muestras para Prueba (Test): 96
--------------------------------------------------


ValueError: Found input variables with inconsistent numbers of samples: [1, 384]